# Save validation U-Net predictions

This notebook saves raw full-field U-Net prediction masks beside the validation images and masks in `data/val_images`. These masks are used as inputs for postprocessing tuning.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/satellite_trails
!pip install -e .[dev]


In [ ]:
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/satellite_trails')
VALIDATION_DIR = REPO_DIR / 'data/val_images'
UNET_CHECKPOINT = REPO_DIR / 'results/models/unet/unet_weights.pt'

UNET_THRESHOLD = 0.65
NORMALIZATION = 'source_zscore'
PATCH_DIM = 528
UNET_BATCH_SIZE = None
NUM_WORKERS = 2
DEVICE = None


In [ ]:
import sys

import numpy as np
import torch
from PIL import Image

sys.path.insert(0, str(REPO_DIR / 'scripts/python'))
from evaluate_final_full_field import build_model
from segmentation import SatelliteTrailPipeline
from satellite_trail_segmentation.unet_model.unet import UNet

Image.MAX_IMAGE_PIXELS = 150000000

device = torch.device(DEVICE) if DEVICE is not None else None
unet_model = build_model(
    UNet,
    UNET_CHECKPOINT,
    ('in_channels', 'out_channels', 'kernel_size', 'base_channels', 'dropout', 'use_batchnorm'),
)
pipeline = SatelliteTrailPipeline(
    unet_model,
    classifier_model=None,
    patch_dim=PATCH_DIM,
    device=device,
    unet_batch_size=UNET_BATCH_SIZE,
    num_workers=NUM_WORKERS,
)

image_paths = sorted(VALIDATION_DIR.glob('*.fits_full.png'))
if not image_paths:
    raise ValueError(f'No validation images found in {VALIDATION_DIR}')
print(f'Found {len(image_paths)} validation fields')


In [ ]:
for index, image_path in enumerate(image_paths, start=1):
    prefix = image_path.name.removesuffix('.fits_full.png')
    mask_path = VALIDATION_DIR / f'{prefix}_mask.png'
    if not mask_path.exists():
        raise FileNotFoundError(f'Missing mask for {image_path.name}: {mask_path.name}')

    print(f'{index}/{len(image_paths)} {image_path.name}')
    preprocessing = pipeline.preprocessing(image_path, mask_path, normalization=NORMALIZATION)
    unet_data, _ = pipeline.segmentation(
        preprocessing['patch_data'],
        use_classifier=False,
        unet_threshold=UNET_THRESHOLD,
    )
    prediction = ((unet_data['segmented_result'] > 0).astype(np.uint8) * 255)
    prediction_path = VALIDATION_DIR / f'{prefix}_prediction.png'
    Image.fromarray(prediction).save(prediction_path)

print(f'Saved predictions to {VALIDATION_DIR}')
